# ViSceT5 — Pretrain **gen_all** (decoder read-scene-text, đòn bẩy #1)
Chạy tuần tự. `gen_all` = huấn luyện decoder **sinh scene-text** (khớp đúng đường finetune: encoder chỉ nhận câu hỏi + ảnh + OCR-feature) + MLM/ITM/TWC làm phụ trợ (×0.5) — phần pretrain trực tiếp có ích cho bộ sinh câu trả lời seq2seq.

Sau khi pretrain xong & upload lên HF, dùng `notebooks/finetune_colab.ipynb` để finetune từ nó.

In [ ]:
!git clone https://github.com/Kussssssss/ViSceT5.git
%cd ViSceT5
# QUAN TRỌNG: các thay đổi pretrain (gen_all, vision unfreeze, whole-word mask) nằm ở
# NHÁNH exp/pretrain-gen-all — KHÔNG phải main. Không checkout đúng nhánh sẽ bị lỗi
# 'unrecognized arguments: --vision_unfreeze_last_n --mlm_mask_mode'.
!git fetch origin
!git checkout exp/pretrain-gen-all
!git pull origin exp/pretrain-gen-all
!git log --oneline -1

In [ ]:
%%capture
!bash setup.sh

In [ ]:
import os
HF_PRETRAIN_REPO = 'Kus669/ViSceT5-pretrain-genall'     # repo sẽ lưu MODEL PRETRAIN (gen_all)

In [ ]:
import argparse
from scripts import prepare_dataset
prepare_dataset.main(argparse.Namespace(config='configs/data/ViTextVQA.yaml', data_dir='./datasets'))

In [ ]:
from scripts import init_model
init_model.main()

### 1) SMOKE / MOCK — TỰ ĐỘNG in debug đầy đủ (không cần set env)
> ⚠️ Vision unfreeze TẮT, có guard NaN pretrain-only. ✅ MLM chỉ dùng câu hỏi. 📖 Gen = **read-scene-text** (denoise đã bỏ).

Mock **luôn** in debug. Trong log tìm:
1. `>>> [pretrain] ... gen = read-scene-text`
2. `🔬 [VERIFY]` toàn ✅, `✅ [GEN] gen_loss finite & > 0`, KHÔNG có `🚨 ... has NaN`
3. Per-step `[Pretrain] ... Loss(M) Loss(I) Loss(TWC) Loss(GEN)` **giảm dần**
4. `🔎 [MLM DEBUG]`: mỗi từ mask hiện **token thô gold/pred + gộp thành word**
5. `🔧 [GEN DEBUG]`: target (OCR reading) vs output

In [ ]:
import importlib
from training import pretrain
importlib.reload(pretrain)
# MOCK: tự động in debug đầy đủ (không cần set env). Full run mặc định KHÔNG debug.
pretrain.main(args_list=[
    'configs/pretrain.yaml',
    '--loss_ablation_mode', 'gen_all',
    '--vision_unfreeze_last_n', '0',   # TẮT tạm: vision unfreeze gây grad-nổ + NaN forward
    '--mlm_mask_mode', 'wholeword',    # mask trọn từ (bỏ 'nạng' copy subword)
    '--smoke_test', 'True',
])

### 2) FULL PRETRAIN — mặc định KHÔNG debug (chỉ progress bar + eval)
Full run **mặc định tắt debug** (chỉ thanh tiến trình train + kết quả eval trên val → tránh đầy output/lag). Muốn **bật debug** cho full: đặt `os.environ['TWC_TRAIN_LOG']='1'` trước khi gọi. Theo dõi `loss_mlm` & `loss_gen` **giảm dần** qua các lần eval.

In [ ]:
import os, importlib
# FRESH constant-LR run, num_train_epochs=10 CỐ ĐỊNH (điều kiện để true-resume = train liền mạch).
# 10 epoch ~20h > 1 phiên Colab -> sẽ chạy theo chặng + true-resume ở Cell 2b.
# Kiểm tra ở ~epoch 5: lấy checkpoint trung gian đem finetune, KHÔNG đổi config.
os.environ['TWC_ADV_PROB']='0.6'; os.environ['TWC_DUP_BOX']='0'
os.environ['MLM_RAND_PROB']='0.25'; os.environ['ITM_WEIGHT']='0'
os.environ.pop('TWC_TRAIN_LOG', None)
from training import pretrain
importlib.reload(pretrain)
pretrain.main(args_list=[
    'configs/pretrain.yaml',
    '--loss_ablation_mode', 'gen_all',
    '--vision_unfreeze_last_n', '0',
    '--mlm_mask_mode', 'wholeword',
    '--num_train_epochs', '10',   # CỐ ĐỊNH 10 (đừng đổi) -> resume sau = giống train liền mạch
])
# Mỗi 1050 step tự lưu checkpoint. Hết phiên/hết giờ -> chạy Cell 3 (upload) rồi Cell 2b (resume).

### 2b) TRUE-RESUME — tiếp tục ĐÚNG như train liền mạch tới 10 epoch
Điều kiện: run gốc đã đặt **`num_train_epochs=10` + constant LR**.
TRUE-resume khôi phục **optimizer + scheduler + RNG + data-skip** → chạy tiếp y như chưa dừng.
- **Giữ `num_train_epochs=10`** (KHÔNG đổi).
- `REPO` = repo checkpoint của **run constant-10 này** (đừng lẫn checkpoint cosine-3 cũ).
- Lặp lại cell này sau mỗi lần Colab ngắt cho tới khi đủ 10 epoch.

In [ ]:
# === TRUE-RESUME: tiếp tục run constant-10 (giống hệt train liền mạch) ===
import os, importlib
from huggingface_hub import list_repo_files, snapshot_download

REPO = 'Kus669/ViSceT5-pretrain-genall-c10'   # repo của RUN CONSTANT-10 (tạo mới, tách khỏi cosine-3 cũ)
OUT  = '/content/ViSceT5/output/pretrain'

files = list_repo_files(REPO)
ckpts = sorted({f.split('/')[0] for f in files if f.startswith('checkpoint-')},
               key=lambda x: int(x.split('-')[1]))
assert ckpts, 'Repo chưa có checkpoint-* (chạy Cell 9 trước rồi Cell 3 upload).'
latest = ckpts[-1]
# TRUE-resume cần FULL checkpoint (optimizer/scheduler/trainer_state/rng)
snapshot_download(REPO, repo_type='model', allow_patterns=[f'{latest}/*'], local_dir=OUT)
resume_path = os.path.join(OUT, latest)
print('True-resume từ:', resume_path, '| files:', sorted(os.listdir(resume_path)))

os.environ['TWC_ADV_PROB']='0.6'; os.environ['TWC_DUP_BOX']='0'
os.environ['MLM_RAND_PROB']='0.25'; os.environ['ITM_WEIGHT']='0'
os.environ.pop('TWC_TRAIN_LOG', None)

from training import pretrain; importlib.reload(pretrain)
pretrain.main(args_list=[
    'configs/pretrain.yaml','--loss_ablation_mode','gen_all',
    '--vision_unfreeze_last_n','0','--mlm_mask_mode','wholeword',
    '--num_train_epochs','10',                # GIỮ NGUYÊN 10 (đúng như run gốc)
    '--resume_from_checkpoint', resume_path,
])
# XONG chặng này -> Cell 3 (upload) để lần sau resume tiếp.

### 3) Upload model pretrain lên HF (để finetune_colab.ipynb dùng)

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
api.create_repo(repo_id=HF_PRETRAIN_REPO, repo_type='model', exist_ok=True)
api.upload_folder(folder_path='/content/ViSceT5/output/pretrain', repo_id=HF_PRETRAIN_REPO,
                  repo_type='model', ignore_patterns=['optimizer.pt'])
print('Uploaded pretrain ->', HF_PRETRAIN_REPO)